# misen + LangGraph 통합 예시

misen 블록을 LangGraph 노드 안에서 사용하는 방법.

핵심: misen 블록은 `dict → dict`이고, LangGraph 노드도 `State → dict`이므로 자연스럽게 연결됩니다.

## 설치

노트북은 레포 루트에서 실행하세요 (`cd misen && jupyter notebook`). 로컬 소스를 editable 모드로 설치합니다.

In [ ]:
%pip install -e ".[dev]" langgraph --quiet

## 1. misen 블록 정의

먼저 재사용 가능한 블록들을 정의합니다. 이 블록들은 LangGraph 없이도 독립적으로 동작합니다.

In [ ]:
from misen import tool, sequential, parallel, branch


@tool
def parse_document(input: dict) -> dict:
    """문서를 파싱해서 텍스트를 추출한다."""
    # 실제로는 파일을 읽겠지만, 예시에서는 시뮬레이션
    file_path = input["file_path"]
    return {
        "text": f"Parsed content from {file_path}",
        "doc_type": "report" if "report" in file_path else "memo",
    }


@tool
def chunk_text(input: dict) -> dict:
    """텍스트를 청크로 나눈다."""
    text = input["text"]
    # 간단한 청킹 시뮬레이션
    chunks = [text[i:i+100] for i in range(0, len(text), 80)]
    return {"chunks": chunks}


@tool
def embed_chunks(input: dict) -> dict:
    """청크를 임베딩으로 변환한다."""
    chunks = input["chunks"]
    # 실제로는 임베딩 모델 호출
    vectors = [[0.1 * i] * 3 for i, _ in enumerate(chunks)]
    return {"vectors": vectors}


@tool
def extract_keywords(input: dict) -> dict:
    """키워드를 추출한다."""
    text = input["text"]
    keywords = text.split()[:5]
    return {"keywords": keywords}


@tool
def generate_summary(input: dict) -> dict:
    """요약을 생성한다."""
    text = input["text"]
    return {"summary": f"Summary of: {text[:50]}..."}

## 2. misen만으로 실행 (LangGraph 없이)

블록들을 조합해서 파이프라인을 만들고, 바로 실행할 수 있습니다.

In [ ]:
# 인제스트 파이프라인: 파싱 → 청킹 → 임베딩
ingest = parse_document | chunk_text | embed_chunks

# 분석 파이프라인: 키워드 + 요약을 병렬로
analyze = parallel(extract_keywords, generate_summary)

# 전체 파이프라인
full_pipeline = sequential(ingest, analyze)

# 실행
result = full_pipeline.run_sync({"file_path": "report_2024.hwp"})

print("chunks:", len(result["chunks"]))
print("vectors:", len(result["vectors"]))
print("keywords:", result["keywords"])
print("summary:", result["summary"])
print("doc_type:", result["doc_type"])

## 3. LangGraph에서 misen 블록 사용하기

misen 블록을 LangGraph 노드 안에서 호출하면 됩니다.

### 방법 A: 블록을 노드 함수 안에서 직접 호출

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# LangGraph State 정의
class DocState(TypedDict, total=False):
    file_path: str
    text: str
    doc_type: str
    chunks: list[str]
    vectors: list[list[float]]
    keywords: list[str]
    summary: str
    decision: str
    analysis_type: str
    result: str


# 노드 안에서 misen 블록 호출
async def ingest_node(state: DocState) -> dict:
    """misen의 ingest 파이프라인을 LangGraph 노드로 사용."""
    return await ingest.run(dict(state))


async def analyze_node(state: DocState) -> dict:
    """misen의 analyze 파이프라인을 LangGraph 노드로 사용."""
    return await analyze.run(dict(state))


async def decide_node(state: DocState) -> dict:
    """문서 타입에 따라 결정."""
    if state["doc_type"] == "report":
        return {"decision": "detailed_review"}
    return {"decision": "quick_scan"}

In [ ]:
# 그래프 구성
graph = StateGraph(DocState)

graph.add_node("ingest", ingest_node)
graph.add_node("analyze", analyze_node)
graph.add_node("decide", decide_node)

graph.add_edge(START, "ingest")
graph.add_edge("ingest", "analyze")
graph.add_edge("analyze", "decide")
graph.add_edge("decide", END)

app = graph.compile()

# 실행
result = await app.ainvoke({"file_path": "report_2024.hwp"})
print(result)

### 방법 B: 헬퍼 함수로 변환을 깔끔하게

반복되는 `dict(state)` 변환을 헬퍼로 빼면 더 깔끔합니다.

In [ ]:
from misen import Block


def as_node(block: Block):
    """misen Block을 LangGraph 노드 함수로 변환."""
    async def node(state: dict) -> dict:
        return await block.run(dict(state))
    node.__name__ = block.name
    return node


# 사용
graph2 = StateGraph(DocState)

graph2.add_node("ingest", as_node(ingest))
graph2.add_node("analyze", as_node(analyze))
graph2.add_node("decide", decide_node)

graph2.add_edge(START, "ingest")
graph2.add_edge("ingest", "analyze")
graph2.add_edge("analyze", "decide")
graph2.add_edge("decide", END)

app2 = graph2.compile()
result = await app2.ainvoke({"file_path": "memo_draft.txt"})
print(result)

## 4. 왜 이렇게 쓰나?

### LangGraph만 쓰면 안 되나?

됩니다. 하지만 misen을 끼우면:

1. **같은 블록을 다른 프로젝트에서 재사용** — `ingest` 파이프라인을 프로젝트 A의 LangGraph에서도, 프로젝트 B의 스크립트에서도 그대로 씀
2. **LangGraph 없이도 동작** — 테스트할 때, 스크립트로 돌릴 때, FastAPI에서 쓸 때 LangGraph 의존성 불필요
3. **LangGraph 노드 안에서 복잡한 조합** — 하나의 노드 안에서 `sequential`, `parallel`, `branch`를 자유롭게 조합

```python
# LangGraph 노드 하나 안에서 복잡한 misen 파이프라인 실행
complex_pipeline = sequential(
    parse_document,
    branch(
        lambda d: d["doc_type"] == "report",
        sequential(chunk_text, embed_chunks),  # report: 청킹+임베딩
        extract_keywords,                       # memo: 키워드만
    ),
    generate_summary,
)
```

이 전체가 LangGraph에서는 **하나의 노드**로 동작합니다.

## 5. 조건 분기가 있는 LangGraph + misen

LangGraph의 conditional edge와 misen의 branch를 함께 쓰는 예시.

In [ ]:
@tool
def detailed_analysis(input: dict) -> dict:
    """상세 분석: 청킹 + 임베딩 + 키워드 + 요약."""
    # 실제로는 이렇게 조합
    return {"analysis_type": "detailed", "result": f"Detailed analysis of {input['text'][:30]}"}


@tool
def quick_analysis(input: dict) -> dict:
    """빠른 분석: 키워드만."""
    return {"analysis_type": "quick", "result": f"Quick scan of {input['text'][:30]}"}


# misen으로 분기 로직을 정의
smart_analysis = branch(
    lambda d: d.get("doc_type") == "report",
    detailed_analysis,
    quick_analysis,
)

# LangGraph에서 사용
graph3 = StateGraph(DocState)

graph3.add_node("parse", as_node(parse_document))
graph3.add_node("smart_analysis", as_node(smart_analysis))  # misen branch가 통째로 노드 하나

graph3.add_edge(START, "parse")
graph3.add_edge("parse", "smart_analysis")
graph3.add_edge("smart_analysis", END)

app3 = graph3.compile()

# report → detailed
r1 = await app3.ainvoke({"file_path": "report_2024.hwp"})
print("report:", r1["analysis_type"])  # detailed

# memo → quick
r2 = await app3.ainvoke({"file_path": "memo_draft.txt"})
print("memo:", r2["analysis_type"])  # quick

## 정리

| 패턴 | 코드 |
|---|---|
| 블록을 노드로 | `graph.add_node("name", as_node(block))` |
| 파이프라인을 노드로 | `graph.add_node("name", as_node(parse \| chunk \| embed))` |
| 분기를 노드로 | `graph.add_node("name", as_node(branch(cond, a, b)))` |
| 블록만 단독 실행 | `result = block.run_sync({...})` |

`as_node`는 3줄짜리 헬퍼입니다. 어댑터 레이어가 아니라 그냥 함수 호출이에요.